In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import seaborn as sns
sns.set_context('notebook', font_scale=1.5, rc={'lines.markeredgewidth': 2})

In [2]:
%load_ext autoreload
%autoreload 2

%matplotlib inline

In [3]:
import matplotlib
print('matplotlib:', matplotlib.__version__)
import numpy
print('numpy:', numpy.__version__)
import scipy
print('scipy:', scipy.__version__)
import pandas 
print('pandas:', pandas.__version__)
import seaborn
print('seaborn:', seaborn.__version__)
import allensdk
print('allensdk:', allensdk.__version__)

matplotlib: 3.4.2
numpy: 1.22.0
scipy: 1.4.1
pandas: 1.5.3
seaborn: 0.13.0
allensdk: 2.16.2


In [4]:
# matplotlib: 3.4.2
# numpy: 1.22.0
# scipy: 1.4.1
# pandas: 1.5.3
# seaborn: 0.13.0
# allensdk: 2.16.2

In [5]:
import visual_behavior.data_access.utilities as utilities
from visual_behavior.data_access import loading as loading

import visual_behavior.visualization.utils as utils
import visual_behavior.visualization.ophys.platform_paper_figures as ppf
import visual_behavior.visualization.ophys.platform_single_cell_examples as pse

from allensdk.brain_observatory.behavior.behavior_project_cache import VisualBehaviorOphysProjectCache

In [6]:
experience_levels = utils.get_new_experience_levels()
experience_level_colors = utils.get_experience_level_colors()
cell_types = utils.get_cell_types()

### load experiments and cells tables

In [7]:
loading.get_platform_analysis_cache_dir()

'\\\\allen\\programs\\braintv\\workgroups\\nc-ophys\\visual_behavior\\platform_paper_cache_new'

In [ ]:
from allensdk.brain_observatory.behavior.behavior_project_cache import VisualBehaviorOphysProjectCache

platform_cache_dir = loading.get_platform_analysis_cache_dir()
cache = VisualBehaviorOphysProjectCache.from_local_cache(cache_dir=platform_cache_dir, use_static_cache=True)
experiments_table = cache.get_ophys_experiment_table()
print(len(experiments_table))

1936


c:\users\marinag\documents\code\allensdk\allensdk\brain_observatory\behavior\behavior_project_cache\behavior_project_cache.py:135: UpdatedStimulusPresentationTableWarning: 
	As of AllenSDK version 2.16.0, the latest Visual Behavior Ophys data has been significantly updated from previous releases. Specifically the user will need to update all processing of the stimulus_presentations tables. These tables now include multiple stimulus types delineated by the columns `stimulus_block` and `stimulus_block_name`.

The data that was available in previous releases are stored in the block name containing 'change_detection' and can be accessed in the pandas table by using: 
	`stimulus_presentations[stimulus_presentations.stimulus_block_name.str.contains('change_detection')]`
  warnings.warn(


In [9]:
# metadata tables
experiments_table = pd.read_csv(os.path.join(platform_cache_dir, 'all_ophys_experiments_table.csv'), index_col=0)
platform_experiments = pd.read_csv(os.path.join(platform_cache_dir, 'platform_paper_ophys_experiments_table.csv'), index_col=0)
platform_cells_table = pd.read_csv(os.path.join(platform_cache_dir, 'platform_paper_ophys_cells_table.csv'), index_col=0)
matched_cells_table = pd.read_csv(os.path.join(platform_cache_dir, 'platform_paper_matched_ophys_cells_table.csv'), index_col=0)

# get lists of matched cells and expts
matched_cells = matched_cells_table.cell_specimen_id.unique()
matched_experiments = matched_cells_table.ophys_experiment_id.unique()

# get cre_lines and cell types for plot labels
cre_lines = np.sort(platform_cells_table.cre_line.unique())
cell_types = utils.get_cell_types_dict(cre_lines, platform_experiments)


In [10]:
print(len(experiments_table.mouse_id.unique()), 'mice')
print(len(experiments_table.ophys_session_id.unique()), 'sessions')
print(len(experiments_table.ophys_container_id.unique()), 'containers')
print(len(experiments_table.index.unique()), 'experiments')

107 mice
703 sessions
326 containers
1936 experiments


In [11]:
print(len(platform_experiments.mouse_id.unique()), 'mice')
print(len(platform_experiments.ophys_session_id.unique()), 'sessions')
print(len(platform_experiments.ophys_container_id.unique()), 'containers')
print(len(platform_experiments.index.unique()), 'experiments')

66 mice
202 sessions
134 containers
402 experiments


### get useful info

In [12]:
palette = utilities.get_experience_level_colors()

In [13]:
save_dir = r'\\allen\programs\braintv\workgroups\nc-ophys\visual_behavior\platform_paper_figures_final\PaperRevision_Neuron\figure_2'

folder = 'population_activity'

## Load response metrics

In [14]:
import visual_behavior.ophys.response_analysis.cell_metrics as cm
import visual_behavior.dimensionality_reduction.clustering.processing as processing

c:\Users\marinag\Anaconda3\envs\visual_behavior_sdk_new\lib\site-packages\statsmodels\compat\pandas.py:61: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  from pandas import Int64Index as NumericIndex
c:\users\marinag\documents\code\allensdk\allensdk\brain_observatory\behavior\behavior_project_cache\behavior_project_cache.py:135: UpdatedStimulusPresentationTableWarning: 
	As of AllenSDK version 2.16.0, the latest Visual Behavior Ophys data has been significantly updated from previous releases. Specifically the user will need to update all processing of the stimulus_presentations tables. These tables now include multiple stimulus types delineated by the columns `stimulus_block` and `stimulus_block_name`.

The data that was available in previous releases are stored in the block name containing 'change_detection' and can be accessed in the pandas table by using: 
	`stimulus_presentat

#### Merged table of cell metrics across conditions

In [15]:

data_type = 'filtered_events'

session_subset = 'full_session'
    
inclusion_criteria = 'platform_experiment_table'

filtered_events_metrics_table = processing.generate_merged_table_of_model_free_metrics(data_type=data_type, session_subset=session_subset,
                                                inclusion_criteria=inclusion_criteria, save_dir=platform_cache_dir)

original_metrics_table = filtered_events_metrics_table.copy()
metrics_table = filtered_events_metrics_table.copy()
len(metrics_table.ophys_experiment_id.unique())


filepath: \\allen\programs\braintv\workgroups\nc-ophys\visual_behavior\platform_paper_cache_new\merged_model_free_metrics_table_filtered_events_platform_experiment_table.csv
loading model free metrics table


402

In [16]:
print(len(metrics_table.cell_specimen_id.unique()), 'cells')
print(len(metrics_table.ophys_experiment_id.unique()), 'experiments')

14495 cells
402 experiments


In [17]:
platform_experiments = loading.get_platform_paper_experiment_table(limit_to_closest_active=True)

c:\users\marinag\documents\code\allensdk\allensdk\brain_observatory\behavior\behavior_project_cache\behavior_project_cache.py:135: UpdatedStimulusPresentationTableWarning: 
	As of AllenSDK version 2.16.0, the latest Visual Behavior Ophys data has been significantly updated from previous releases. Specifically the user will need to update all processing of the stimulus_presentations tables. These tables now include multiple stimulus types delineated by the columns `stimulus_block` and `stimulus_block_name`.

The data that was available in previous releases are stored in the block name containing 'change_detection' and can be accessed in the pandas table by using: 
	`stimulus_presentations[stimulus_presentations.stimulus_block_name.str.contains('change_detection')]`
  warnings.warn(


removing 1 problematic experiments


In [18]:
platform_experiments.experience_level.unique()

array(['Familiar', 'Novel', 'Novel +'], dtype=object)

In [19]:
print(len(platform_experiments.mouse_id.unique()), 'mice')
print(len(platform_experiments.ophys_session_id.unique()), 'sessions')
print(len(platform_experiments.ophys_container_id.unique()), 'containers')
print(len(platform_experiments.index.unique()), 'experiments')

66 mice
202 sessions
134 containers
402 experiments


In [20]:
metrics_table = metrics_table.merge(platform_experiments.reset_index(), on=['ophys_experiment_id', 'experience_level'])
print(len(metrics_table.ophys_experiment_id.unique()))
print(len(metrics_table.cell_specimen_id.unique()))

402
14495


In [21]:
print(len(metrics_table.mouse_id.unique()), 'mice')
print(len(metrics_table.ophys_session_id.unique()), 'sessions')
print(len(metrics_table.ophys_container_id.unique()), 'containers')
print(len(metrics_table.ophys_experiment_id.unique()), 'experiments')
print(len(metrics_table.cell_specimen_id.unique()), 'cells')

66 mice
202 sessions
134 containers
402 experiments
14495 cells


## Attempt hierarchical stats with mixed linear model

In [22]:
import statsmodels.formula.api as smf
from statsmodels.formula.api import ols
import statsmodels.api as sm

np.random.seed(42)

#### Get metric data to perform stats on

In [23]:
metrics_table.head()

,cell_specimen_id,experience_level,ophys_experiment_id,mean_response_pref_image,reliability_pref_image,fraction_significant_p_value_gray_screen_pref_image,fano_factor_pref_image,running_modulation_pref_image,mean_response_all_images,lifetime_sparseness_images,...,layer,area_layer,date,first_novel,n_relative_to_first_novel,last_familiar,last_familiar_active,second_novel,second_novel_active,experience_exposure
0,1086490480,Familiar,795073741,0.000131,0.122168,0.011364,0.011834,1.0,0.000066,0.348866,...,upper,VISp_upper,20181213,False,-1.0,True,True,False,False,Familiar 3
1,1086490397,Familiar,795073741,0.000961,0.452276,0.000000,0.019686,1.0,0.000396,0.424478,...,upper,VISp_upper,20181213,False,-1.0,True,True,False,False,Familiar 3
2,1086490510,Familiar,795073741,0.004260,0.183585,0.160149,0.043784,1.0,0.001107,0.574101,...,upper,VISp_upper,20181213,False,-1.0,True,True,False,False,Familiar 3
3,1086490565,Familiar,795073741,0.000229,0.738372,0.002410,0.013540,1.0,0.000092,0.374332,...,upper,VISp_upper,20181213,False,-1.0,True,True,False,False,Familiar 3
4,1086490680,Familiar,795073741,0.003442,0.336522,0.072289,0.082742,1.0,0.001172,0.441491,...,upper,VISp_upper,20181213,False,-1.0,True,True,False,False,Familiar 3


In [24]:
metrics_table.columns

Index(['cell_specimen_id', 'experience_level', 'ophys_experiment_id',
       'mean_response_pref_image', 'reliability_pref_image',
       'fraction_significant_p_value_gray_screen_pref_image',
       'fano_factor_pref_image', 'running_modulation_pref_image',
       'mean_response_all_images', 'lifetime_sparseness_images',
       'reliability_all_images',
       'fraction_significant_p_value_gray_screen_all_images',
       'fano_factor_all_images', 'running_modulation_all_images',
       'mean_response_omissions', 'reliability_omissions',
       'fraction_significant_p_value_gray_screen_omissions',
       'fano_factor_omissions', 'running_modulation_omissions',
       'omission_modulation_index', 'mean_response_changes',
       'mean_response_pre_change', 'lifetime_sparseness_changes',
       'reliability_changes',
       'fraction_significant_p_value_gray_screen_changes',
       'fano_factor_changes', 'running_modulation_changes',
       'change_modulation_index', 'hit_miss_index', 'be

In [25]:
metric = 'mean_response_pref_image'
data = metrics_table[['cell_specimen_id',  'mouse_id', 'experience_level', 'cell_type']+[metric]]
data = data[data.cell_type=='Excitatory']

print(len(data.cell_specimen_id.unique()))
data.head()

12826


,cell_specimen_id,mouse_id,experience_level,cell_type,mean_response_pref_image
127,1086492747,425493,Familiar,Excitatory,0.000615
128,1086493116,425493,Familiar,Excitatory,0.000213
129,1086496725,425493,Familiar,Excitatory,0.000262
130,1086496929,425493,Familiar,Excitatory,0.000331
131,1086497365,425493,Familiar,Excitatory,0.012457


#### run mixed linear model with random intercept to check if mouse ID explains variance beyond experience level

In [26]:
# --- 2. Run the Mixed Effects Model (Random Intercept) ---
print("\n--- Running HLM (Random Intercept) ---")

formula = metric+' ~ experience_level'
# Define the model: Test_Score ~ Study_Hours, with a random intercept for Classroom_ID
model_ri = smf.mixedlm(
    formula=formula,
    data=data,
    groups='mouse_id',
    vc_formula={'mouse_id': '0 + C(mouse_id)'} # Specify random intercept
)

# Fit the model
result_ri = model_ri.fit()

# Print the results summary
print(result_ri.summary())


--- Running HLM (Random Intercept) ---
                Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: mean_response_pref_image
No. Observations: 22680   Method:             REML                    
No. Groups:       34      Scale:              0.0085                  
Min. group size:  138     Log-Likelihood:     21795.2965              
Max. group size:  2807    Converged:          Yes                     
Mean group size:  667.1                                               
----------------------------------------------------------------------
                              Coef. Std.Err.   z   P>|z| [0.025 0.975]
----------------------------------------------------------------------
Intercept                     0.004    0.002 2.399 0.016  0.001  0.007
experience_level[T.Novel]     0.006    0.002 3.892 0.000  0.003  0.009
experience_level[T.Novel +]   0.003    0.002 2.051 0.040  0.000  0.006
mouse_id Var                  0.000    0.000          

c:\Users\marinag\Anaconda3\envs\visual_behavior_sdk_new\lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


* scale = This is the Level 1 residual variance (σ2), representing the unexplained variation within a mouse_id (the individual-level error).
* REML - Restricted Maximum Likelihood, the standard fitting method for estimating variance components (random effects).
* Reference category is Familiar, here listed as "Intercept"
* Coef. is the predicted average mean response
* P>z is the p value

* All fixed effects, including the reference intercept and the effect of both "Novel" and "Novel +" experience levels, are statistically significant predictors of the mean response changes. Both "Novel" groups are associated with a slight increase in the outcome compared to the baseline.
* The random effects describe how much the intercepts vary across your groups
* Coef. for mouse_id Var is 0, meaning the estimated variance of the intercepts across the 34 mouse_id groups is zero
* This is a very important finding. It suggests that, after accounting for the experience_level, the baseline mean_response_changes does not significantly vary between individual mice. In other words, the mice are essentially starting at the same point
* The finding that the random intercept variance is zero suggests that while your data is technically nested (multiple observations per mouse), the variation in the outcome is almost entirely at the individual (Level 1) level (Scale = 0.0004) and not at the group (Level 2) level.
* Because the group-level variance (τ02​) is zero, the hierarchical linear model structure is not strictly necessary to handle non-independence, as the independence assumption is not strongly violated in this case. You could potentially use a simpler standard linear regression (OLS), and the results for the fixed effects would be very similar.

Term | Coef. | P>|z| | Interpretation | | :--- | :--- | :--- | :--- | | Intercept | 0.002 | 0.000 | This is the predicted average mean_response_changes for the reference category of experience_level. Since the other categories are "Novel" and "Novel +", the reference category is likely a baseline group (e.g., "Experienced" or "Control"). This baseline score is statistically significant. | | experience_level[T.Novel] | 0.001 | 0.000 | Mice with the "Novel" experience level have a mean_response_changes that is 0.001 units higher than the reference group, on average. This difference is statistically significant. | | experience_level[T.Novel +] | 0.001 | 0.048 | Mice with the "Novel +" experience level have a mean_response_changes that is also 0.001 units higher than the reference group, on average. This difference is statistically significant (just under the α=0.05 threshold).

#### log likelihood ratio test to compare to a random intercept and slope model

* The most rigorous method is to formally compare a simpler model (Random Intercept Only) with a more complex model (Random Intercept and Random Slope) using a Likelihood Ratio Test (LRT)
* Fit your HLM model using Maximum Likelihood (ML) estimation (not REML, which is the default for many software packages like statsmodels for final results, but required for model comparison

1. Fit the Simpler Model (Random Intercept): Fit your HLM model using Maximum Likelihood (ML) estimation (not REML, which is the default for many software packages like statsmodels for final results, but required for model comparison).
    Model A: Outcome∼Predictor+(1∣Group)
2. Fit the Complex Model (Random Slope): Fit the same model, but add the random slope for the predictor of interest.
    Model B: Outcome∼Predictor+(Predictor∣Group)
3. Perform the LRT: The test compares the fit of the two nested models by calculating the difference in their log-likelihoods.
    χ2=−2×(Log-LikelihoodModel A​−Log-LikelihoodModel B​)


interpret: 
* Significant p-value (p<0.05): This means the more complex model (Model B, with the random slope) provides a significantly better fit to the data than the simpler model. You should include the random slope. 
* Non-Significant p-value (p≥0.05): The random slope does not add significant explanatory power. You should retain the simpler, more parsimonious Random Intercept Model (Model A).

In [27]:
from statsmodels.formula.api import ols
from scipy.stats import chi2

run the 3 model variants and do the likelihood ratio test to decide which one to use

In [28]:
formula = metric+' ~ experience_level'

##### Fit 3 model types then do likelihood ratio test to figure out which is appropriate

# --- Model A: Fixed Effects Only (OLS equivalent) ---
# This is a model with no random effects.
model_A_fit = ols(formula=formula, data=data).fit()

# --- Model B: Random Intercept (RI) ---
# Random effect for the intercept (1) grouped by Classroom_ID.
model_B = smf.mixedlm(formula=formula, data=data,
    groups=data["mouse_id"],
    vc_formula={'mouse_id': '0 + C(mouse_id)'} # Specify random intercept
)
model_B_fit = model_B.fit(reml=False) # Use ML for LRT

# --- Model C: Random Intercept and Random Slope (RI+RS) ---
# Random effects for both the intercept (1) and the slope (Study_Hours)
model_C = smf.mixedlm(
    formula=formula,
    data=data,
    groups=data["mouse_id"],
    re_formula='~ 1 + experience_level', # Specify both random intercept (1) and random slope
    vc_formula={'mouse_id': '0 + C(mouse_id)'} # Specify random intercept
)
model_C_fit = model_C.fit(reml=False) # Use ML for LRT

print("Models fitted using Maximum Likelihood (ML) estimation.")

c:\Users\marinag\Anaconda3\envs\visual_behavior_sdk_new\lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
c:\Users\marinag\Anaconda3\envs\visual_behavior_sdk_new\lib\site-packages\statsmodels\base\model.py:604: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\marinag\Anaconda3\envs\visual_behavior_sdk_new\lib\site-packages\statsmodels\regression\mixed_linear_model.py:2200: ConvergenceWarning: Retrying MixedLM optimization with lbfgs
  warnings.warn(
c:\Users\marinag\Anaconda3\envs\visual_behavior_sdk_new\lib\site-packages\statsmodels\base\model.py:604: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\marinag\Anaconda3\envs\visual_behav

Models fitted using Maximum Likelihood (ML) estimation.


c:\Users\marinag\Anaconda3\envs\visual_behavior_sdk_new\lib\site-packages\statsmodels\base\model.py:604: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\marinag\Anaconda3\envs\visual_behavior_sdk_new\lib\site-packages\statsmodels\regression\mixed_linear_model.py:2206: ConvergenceWarning: MixedLM optimization failed, trying a different optimizer may help.
  warnings.warn(msg, ConvergenceWarning)
c:\Users\marinag\Anaconda3\envs\visual_behavior_sdk_new\lib\site-packages\statsmodels\regression\mixed_linear_model.py:2218: ConvergenceWarning: Gradient optimization failed, |grad| = 54.413326
  warnings.warn(msg, ConvergenceWarning)
c:\Users\marinag\Anaconda3\envs\visual_behavior_sdk_new\lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
c:\Users\marin

likelihood ratio test to check which one is best

In [29]:
def likelihood_ratio_test(model_restricted, model_full, df_diff):
    """
    Performs a Likelihood Ratio Test (LRT) to compare two nested models.
    
    Args:
        model_restricted: The simpler model (e.g., Random Intercept).
        model_full: The more complex model (e.g., Random Intercept + Slope).
        df_diff: The difference in the number of estimated parameters (degrees of freedom).
    
    Returns:
        A dictionary with LRT statistics.
    """
    # Extract Log-Likelihoods
    LL_restricted = model_restricted.llf
    LL_full = model_full.llf
    
    # Calculate the LRT statistic (Test Statistic)
    LRT_stat = -2 * (LL_restricted - LL_full)
    
    # Calculate the p-value using the chi-squared distribution
    p_value = chi2.sf(LRT_stat, df_diff)
    
    return {
        'LRT Statistic ($\chi^2$)': LRT_stat,
        'Degrees of Freedom (df)': df_diff,
        'P-value': p_value
    }

This tests whether adding the random intercept significantly improves the model fit.

In [30]:
print("\n--- LRT: Random Intercept (B) vs. Fixed Effects (A) ---")
# The LLF for OLS (Model A) is stored as llf, df is 1 (tau_0^2)
lrt_B_vs_A = likelihood_ratio_test(model_A_fit, model_B_fit, df_diff=1)
print(pd.Series(lrt_B_vs_A).to_string())

# Interpretation: If P-value < 0.05, the random intercept is justified.


--- LRT: Random Intercept (B) vs. Fixed Effects (A) ---
LRT Statistic ($\chi^2$)    9.760671e+01
Degrees of Freedom (df)     1.000000e+00
P-value                     5.103068e-23


In [31]:
lrt_B_vs_A['P-value']

5.103068400606049e-23

p-value is microscopic --> use random intercept

means that mice have different starting points?

This tests whether adding the random slope significantly improves the model fit.

In [32]:
print("\n--- LRT: Random Intercept + Slope (C) vs. Random Intercept (B) ---")
# The full model (C) has 2 more random effects parameters than model (B):
# 1. Variance of the random slope (tau_1^2)
# 2. Covariance between the random intercept and random slope (tau_01)
lrt_C_vs_B = likelihood_ratio_test(model_B_fit, model_C_fit, df_diff=2) 
print(pd.Series(lrt_C_vs_B).to_string())

# Interpretation: 
# If P-value < 0.05, the random slope is justified and significantly improves the model fit.
# If P-value >= 0.05, the simpler Random Intercept model (B) is preferred.


--- LRT: Random Intercept + Slope (C) vs. Random Intercept (B) ---
LRT Statistic ($\chi^2$)   -259.935944
Degrees of Freedom (df)       2.000000
P-value                       1.000000


Looks like random slope is needed too

In [34]:
# --- 2. Run the Mixed Effects Model (Random Intercept) ---
print("\n--- Running HLM (Random Intercept) ---")

formula = metric+' ~ experience_level'

# Define the model: Test_Score ~ Study_Hours, with a random intercept for Classroom_ID
model_ri_rs = smf.mixedlm(
    formula=formula,
    data=data,
    groups='mouse_id',
    re_formula='~ 1 + experience_level'
    # vc_formula={'mouse_id': '0 + C(mouse_id)'} # Specify random intercept
)

# Fit the model
result_ri_rs = model_ri_rs.fit()

# Print the results summary
print(result_ri_rs.summary())


--- Running HLM (Random Intercept) ---


c:\Users\marinag\Anaconda3\envs\visual_behavior_sdk_new\lib\site-packages\statsmodels\base\model.py:604: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\marinag\Anaconda3\envs\visual_behavior_sdk_new\lib\site-packages\statsmodels\regression\mixed_linear_model.py:2200: ConvergenceWarning: Retrying MixedLM optimization with lbfgs
  warnings.warn(


                               Mixed Linear Model Regression Results
Model:                      MixedLM           Dependent Variable:           mean_response_pref_image
No. Observations:           22680             Method:                       REML                    
No. Groups:                 34                Scale:                        0.0085                  
Min. group size:            138               Log-Likelihood:               21804.2104              
Max. group size:            2807              Converged:                    Yes                     
Mean group size:            667.1                                                                   
----------------------------------------------------------------------------------------------------
                                                            Coef. Std.Err.   z   P>|z| [0.025 0.975]
----------------------------------------------------------------------------------------------------
Intercept             

c:\Users\marinag\Anaconda3\envs\visual_behavior_sdk_new\lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
c:\Users\marinag\Anaconda3\envs\visual_behavior_sdk_new\lib\site-packages\statsmodels\regression\mixed_linear_model.py:2261: ConvergenceWarning: The Hessian matrix at the estimated parameter values is not positive definite.
  warnings.warn(msg, ConvergenceWarning)


* The model attempted to fit a very complex random effects structure, but the results strongly suggest that this complexity is not supported by the data.
* Scale	0.0004	The Level 1 (within-mouse) residual variance (σ2). This is the unexplained variance at the individual observation level.

| Term | Coef. | P>|z| | Interpretation | | :--- | :--- | :--- | :--- | | Intercept | 0.002 | 0.000 | This is the predicted average mean_response_changes for the reference category of experience_level (likely the baseline, e.g., "Experienced" or "Control"). This baseline response is statistically significant. | | experience_level[T.Novel] | 0.001 | 0.082 | Mice in the "Novel" group have an average response 0.001 units higher than the reference group. This difference is marginally statistically significant (if you use α=0.10) but not significant at the conventional α=0.05 level. | | experience_level[T.Novel +] | 0.000 | 0.484 | Mice in the "Novel +" group show practically no average difference (0.000) from the reference group. This difference is not statistically significant. |

This tests whether adding the random intercept significantly improves the model fit.

In [35]:
print("\n--- LRT: Random Intercept (B) vs. Fixed Effects (A) ---")
# The LLF for OLS (Model A) is stored as llf, df is 1 (tau_0^2)
lrt_B_vs_A = likelihood_ratio_test(model_A_fit, model_B_fit, df_diff=1)
print(pd.Series(lrt_B_vs_A).to_string())

# Interpretation: If P-value < 0.05, the random intercept is justified.


--- LRT: Random Intercept (B) vs. Fixed Effects (A) ---
LRT Statistic ($\chi^2$)    9.760671e+01
Degrees of Freedom (df)     1.000000e+00
P-value                     5.103068e-23


In [36]:
lrt_B_vs_A['P-value']

5.103068400606049e-23

p-value is microscopic --> use random intercept

This tests whether adding the random slope significantly improves the model fit.

In [37]:
print("\n--- LRT: Random Intercept + Slope (C) vs. Random Intercept (B) ---")
# The full model (C) has 2 more random effects parameters than model (B):
# 1. Variance of the random slope (tau_1^2)
# 2. Covariance between the random intercept and random slope (tau_01)
lrt_C_vs_B = likelihood_ratio_test(model_B_fit, model_C_fit, df_diff=2) 
print(pd.Series(lrt_C_vs_B).to_string())

# Interpretation: 
# If P-value < 0.05, the random slope is justified and significantly improves the model fit.
# If P-value >= 0.05, the simpler Random Intercept model (B) is preferred.


--- LRT: Random Intercept + Slope (C) vs. Random Intercept (B) ---
LRT Statistic ($\chi^2$)   -259.935944
Degrees of Freedom (df)       2.000000
P-value                       1.000000


Looks like random slope is needed too

In [38]:
# --- 2. Run the Mixed Effects Model (Random Intercept) ---
print("\n--- Running HLM (Random Intercept) ---")

formula = metric+' ~ experience_level'

# Define the model: Test_Score ~ Study_Hours, with a random intercept for Classroom_ID
model_ri_rs = smf.mixedlm(
    formula=formula,
    data=data,
    groups='mouse_id',
    re_formula='~ 1 + experience_level'
    # vc_formula={'mouse_id': '0 + C(mouse_id)'} # Specify random intercept
)

# Fit the model
result_ri_rs = model_ri_rs.fit()

# Print the results summary
print(result_ri_rs.summary())


--- Running HLM (Random Intercept) ---


c:\Users\marinag\Anaconda3\envs\visual_behavior_sdk_new\lib\site-packages\statsmodels\base\model.py:604: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\marinag\Anaconda3\envs\visual_behavior_sdk_new\lib\site-packages\statsmodels\regression\mixed_linear_model.py:2200: ConvergenceWarning: Retrying MixedLM optimization with lbfgs
  warnings.warn(


                               Mixed Linear Model Regression Results
Model:                      MixedLM           Dependent Variable:           mean_response_pref_image
No. Observations:           22680             Method:                       REML                    
No. Groups:                 34                Scale:                        0.0085                  
Min. group size:            138               Log-Likelihood:               21804.2104              
Max. group size:            2807              Converged:                    Yes                     
Mean group size:            667.1                                                                   
----------------------------------------------------------------------------------------------------
                                                            Coef. Std.Err.   z   P>|z| [0.025 0.975]
----------------------------------------------------------------------------------------------------
Intercept             

c:\Users\marinag\Anaconda3\envs\visual_behavior_sdk_new\lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
c:\Users\marinag\Anaconda3\envs\visual_behavior_sdk_new\lib\site-packages\statsmodels\regression\mixed_linear_model.py:2261: ConvergenceWarning: The Hessian matrix at the estimated parameter values is not positive definite.
  warnings.warn(msg, ConvergenceWarning)


* The model attempted to fit a very complex random effects structure, but the results strongly suggest that this complexity is not supported by the data.
* Scale	0.0004	The Level 1 (within-mouse) residual variance (σ2). This is the unexplained variance at the individual observation level.

| Term | Coef. | P>|z| | Interpretation | | :--- | :--- | :--- | :--- | | Intercept | 0.002 | 0.000 | This is the predicted average mean_response_changes for the reference category of experience_level (likely the baseline, e.g., "Experienced" or "Control"). This baseline response is statistically significant. | | experience_level[T.Novel] | 0.001 | 0.082 | Mice in the "Novel" group have an average response 0.001 units higher than the reference group. This difference is marginally statistically significant (if you use α=0.10) but not significant at the conventional α=0.05 level. | | experience_level[T.Novel +] | 0.000 | 0.484 | Mice in the "Novel +" group show practically no average difference (0.000) from the reference group. This difference is not statistically significant. |

### Define functions and run for several conditions

In [39]:
def likelihood_ratio_test(model_restricted, model_full, df_diff):
    """
    Performs a Likelihood Ratio Test (LRT) to compare two nested models.
    
    Args:
        model_restricted: The simpler model (e.g., Random Intercept).
        model_full: The more complex model (e.g., Random Intercept + Slope).
        df_diff: The difference in the number of estimated parameters (degrees of freedom).
    
    Returns:
        A dictionary with LRT statistics.
    """
    # Extract Log-Likelihoods
    LL_restricted = model_restricted.llf
    LL_full = model_full.llf
    
    # Calculate the LRT statistic (Test Statistic)
    LRT_stat = -2 * (LL_restricted - LL_full)
    
    # Calculate the p-value using the chi-squared distribution
    p_value = chi2.sf(LRT_stat, df_diff)
    
    return {
        'LRT Statistic ($\chi^2$)': LRT_stat,
        'Degrees of Freedom (df)': df_diff,
        'P-value': p_value
    }

In [40]:

# --- 1. Define the extraction function ---
def extract_mixedlm_summary(model_results):
    """
    Extracts key results from a fitted statsmodels MixedLM model.
    """
    # 1. Extract Fixed Effects Table (The Coef. table)
    fixed_effects_df = model_results.summary().tables[0]
    fixed_effects_dict = fixed_effects_df.to_dict(orient='dict')

    # 2. Extract Random Effects Table (The Variance/Covariance table)
    random_effects_df = model_results.summary().tables[1]
    random_effects_dict = random_effects_df.to_dict(orient='dict')

    random_effects_df = model_results.summary().tables[1]

    # 3. Extract Model Fit and Group Statistics
    model_stats = {
        'Log-Likelihood': model_results.llf,
        'AIC': model_results.aic,
        'BIC': model_results.bic,
        'No. Observations': model_results.nobs,
        'No. Groups': model_results.summary().tables[0].iloc[2][1],
        'Scale_Residual_Variance': model_results.scale 
    }

    # 4. Combine into a Single Dictionary
    full_results_dict = {
        'Model_Type': 'Mixed Linear Model (MixedLM)',
        'Model_Statistics': model_stats,
        'Fixed_Effects': fixed_effects_dict,
        'Random_Effects_Variance': random_effects_dict
    }
    
    return full_results_dict

# --- Example Usage (Assuming model_B_fit is available) ---
# final_dict = extract_mixedlm_summary_corrected(model_B_fit)
# print(json.dumps(final_dict, indent=4, sort_keys=False))

def extract_ols_summary(model_A_fit):

# --- 1. Extract Fixed Effects (Coefficients) ---
    # The params_rslt object contains the Coef., Std.Err., z, P>|z|, and CI bounds

    # The results are stored in several key attributes:
    fixed_effects_df = pd.DataFrame({
        'Coef.': model_A_fit.params,
        'Std. Err.': model_A_fit.bse,
        't': model_A_fit.tvalues,
        'P>|t|': model_A_fit.pvalues,
        # Confidence intervals are stored in ols_results.conf_int()
        'Lower CI': model_A_fit.conf_int()[0],
        'Upper CI': model_A_fit.conf_int()[1]
    })
    # fixed_effects_df = model_A_fit.summary()
    fixed_effects_dict = fixed_effects_df.to_dict(orient='index')

    # --- 2. Extract Model Fit Statistics (R-squared, Log-Likelihood, etc.) ---
    # statsmodels stores many scalar attributes directly on the result object
    model_stats = {
        'R-squared': model_A_fit.rsquared,
        'Adj. R-squared': model_A_fit.rsquared_adj,
        'F-statistic': model_A_fit.fvalue,
        'Prob (F-statistic)': model_A_fit.f_pvalue,
        'Log-Likelihood': model_A_fit.llf,
        'AIC': model_A_fit.aic,
        'BIC': model_A_fit.bic,
        'No. Observations': model_A_fit.nobs
    }


    # --- 3. Combine into a Single Dictionary ---
    full_results_dict = {
        'Model_Type': 'OLS Regression (Fixed Effects Only)',
        'Model_Statistics': model_stats,
        'Fixed_Effects': fixed_effects_dict
    }

    return full_results_dict

In [41]:
def run_mixed_linear_model_stats_for_metric(data, metric): 
    '''
    Function to select and run a mixed linear model for nested stats to account for cells coming from specific mice
    Will test to see if mouse_id explains any residual variance after fitting data to cell metric values across experience levels
    
    data should be a dataframe with columns for the metric value, experience level, and mouse_id
    metric is a string, name of column in data to compute stats for 
    '''    
    formula = metric+' ~ experience_level'

    ##### Fit 3 model types then do likelihood ratio test to figure out which is appropriate

    # --- Model A: Fixed Effects Only (OLS equivalent) ---
    # This is a model with no random effects.
    model_A_fit = ols(formula=formula, data=data).fit()

    # --- Model B: Random Intercept (RI) ---
    # Random effect for the intercept (1) grouped by Classroom_ID.
    model_B = smf.mixedlm(formula=formula, data=data,
        groups=data["mouse_id"],
        vc_formula={'mouse_id': '0 + C(mouse_id)'} # Specify random intercept
    )
    model_B_fit = model_B.fit(reml=False) # Use ML for LRT

    # --- Model C: Random Intercept and Random Slope (RI+RS) ---
    # Random effects for both the intercept (1) and the slope (Study_Hours)
    model_C = smf.mixedlm(
        formula=formula,
        data=data,
        groups=data["mouse_id"],
        re_formula='~ 1 + experience_level', # Specify both random intercept (1) and random slope
        vc_formula={'mouse_id': '0 + C(mouse_id)'} # Specify random intercept
    )
    model_C_fit = model_C.fit(reml=False) # Use ML for LRT

    print("Models fitted using Maximum Likelihood (ML) estimation.")

    #### likelihood ratio test

    print("\n--- LRT: Random Intercept (B) vs. Fixed Effects (A) ---")
    # The LLF for OLS (Model A) is stored as llf, df is 1 (tau_0^2)
    lrt_B_vs_A = likelihood_ratio_test(model_A_fit, model_B_fit, df_diff=1)
    print(pd.Series(lrt_B_vs_A).to_string())
    # Interpretation: If P-value < 0.05, the random intercept is justified.

    print("\n--- LRT: Random Intercept + Slope (C) vs. Random Intercept (B) ---")
    # The full model (C) has 2 more random effects parameters than model (B):
    # 1. Variance of the random slope (tau_1^2)
    # 2. Covariance between the random intercept and random slope (tau_01)
    lrt_C_vs_B = likelihood_ratio_test(model_B_fit, model_C_fit, df_diff=2) 
    print(pd.Series(lrt_C_vs_B).to_string())
    # Interpretation: 
    # If P-value < 0.05, the random slope is justified and significantly improves the model fit.
    # If P-value >= 0.05, the simpler Random Intercept model (B) is preferred.

    #### Decide which to use

    if lrt_C_vs_B['P-value'] < 0.05: # if the p-value for complex model is < alpha
        # then use the random intercept and random slope model
        stats_dict = extract_mixedlm_summary(model_C_fit)
        model_to_use = 'C'
    else: 
        # then check the random intercept only MLS versus OLS
        if lrt_B_vs_A['P-value'] < 0.05: # if the p-value for MLS with random intercept is greater than OLS
            # then use this model
            stats_dict = extract_mixedlm_summary(model_B_fit)
            model_to_use = 'B'
        else: # otherwise use ordinary least squares
            stats_dict = extract_ols_summary(model_A_fit)
            model_to_use = 'A'
    print('\n using model', model_to_use)

    #### Now run the relevant model with REML

    if model_to_use == 'A':
        model_fit = ols(formula=formula, data=data).fit()
        stats_dict = extract_ols_summary(model_fit)

    elif model_to_use == 'B': 
        # --- Model B: Random Intercept (RI) ---
        model_fit = smf.mixedlm(formula=formula,data=data, groups=data["mouse_id"],
            vc_formula={'mouse_id': '0 + C(mouse_id)'}, # Specify random intercept
            ).fit()
        stats_dict = extract_mixedlm_summary(model_fit)
        
    elif model_to_use == 'C':
        # --- Model C: Random Intercept and Random Slope (RI+RS) ---
        # Random effects for both the intercept (1) and the slope (Study_Hours)
        model_fit = smf.mixedlm(formula=formula, data=data, groups=data["mouse_id"],
            vc_formula={'mouse_id': '0 + C(mouse_id)'}, # Specify random intercept
            re_formula='~ 1 + experience_level', # Specify both random intercept (1) and random slope
            ).fit()
        stats_dict = extract_mixedlm_summary(model_fit)

    print('\n')
    print(pd.DataFrame(stats_dict['Fixed_Effects']))
    print('\n')
    if 'Random_Effects_Variance' in stats_dict.keys():
        print(pd.DataFrame(stats_dict['Random_Effects_Variance']))

    return stats_dict


### how to interpret

Scenario: log-likelihood test tells you that you need random effects model (i.e. hierarchical model) because there is variance across subjects (level 2 predictor). how do you know if the results at the individual level (level 1 predictor) are significant? 

Here is the precise interpretation of that scenario:

* You "need" a random effects model: This means the Likelihood Ratio Test confirmed significant between-group variability (the random intercept or slope is necessary). Failing to use a mixed model would likely lead to inaccurate (underestimated) standard errors for your fixed effects.
* The p-value for the Level 1 fixed effect is significant: This p-value, derived from the statsmodels summary table, tells you that there is a statistically reliable, non-zero relationship between that predictor variable (measured at the individual/observation level) and the outcome variable.

In summary: Yes, if you run a correctly specified mixed-effects model and the p-value for your Level 1 predictor is significant, you can confidently conclude that the Level 1 predictor is a significant predictor of your outcome, <b>while appropriately controlling for the dependency among observations within groups</b>.

### Now run it on a few examples

In [48]:
metric = 'mean_response_changes'
tmp = metrics_table[['cell_specimen_id',  'mouse_id', 'experience_level', 'cell_type']+[metric]]
cell_types = utils.get_cell_types()

for cell_type in cell_types: 
    print('\n\n\n'+cell_type+'\n\n\n')
    data = tmp[tmp.cell_type==cell_type]
    stats_dict = run_mixed_linear_model_stats_for_metric(data, metric)




Excitatory





c:\Users\marinag\Anaconda3\envs\visual_behavior_sdk_new\lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
c:\Users\marinag\Anaconda3\envs\visual_behavior_sdk_new\lib\site-packages\statsmodels\base\model.py:604: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\marinag\Anaconda3\envs\visual_behavior_sdk_new\lib\site-packages\statsmodels\regression\mixed_linear_model.py:2200: ConvergenceWarning: Retrying MixedLM optimization with lbfgs
  warnings.warn(
c:\Users\marinag\Anaconda3\envs\visual_behavior_sdk_new\lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
c:\Users\marinag\Anaconda3\envs\visual_behavior_sdk_new\lib

Models fitted using Maximum Likelihood (ML) estimation.

--- LRT: Random Intercept (B) vs. Fixed Effects (A) ---
LRT Statistic ($\chi^2$)    3.744184e+02
Degrees of Freedom (df)     1.000000e+00
P-value                     2.042666e-83

--- LRT: Random Intercept + Slope (C) vs. Random Intercept (B) ---
LRT Statistic ($\chi^2$)   -183.891357
Degrees of Freedom (df)       2.000000
P-value                       1.000000

 using model B


c:\Users\marinag\Anaconda3\envs\visual_behavior_sdk_new\lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)




                   0        1                    2                      3
0             Model:  MixedLM  Dependent Variable:  mean_response_changes
1  No. Observations:    22680              Method:                   REML
2        No. Groups:       34               Scale:                 0.0004
3   Min. group size:      138      Log-Likelihood:             57663.7461
4   Max. group size:     2807           Converged:                    Yes
5   Mean group size:    667.1                                            


                             Coef. Std.Err.      z  P>|z| [0.025 0.975]
Intercept                    0.002    0.000  4.104  0.000  0.001  0.003
experience_level[T.Novel]    0.001    0.000  4.581  0.000  0.001  0.002
experience_level[T.Novel +]  0.001    0.000  1.976  0.048  0.000  0.001
mouse_id Var                 0.000    0.000                            



Sst Inhibitory





...

c:\Users\marinag\Anaconda3\envs\visual_behavior_sdk_new\lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)




                   0        1                    2                      3
0             Model:  MixedLM  Dependent Variable:  mean_response_changes
1  No. Observations:      978              Method:                   REML
2        No. Groups:       15               Scale:                 0.0018
3   Min. group size:       11      Log-Likelihood:              1689.3814
4   Max. group size:      234           Converged:                    Yes
5   Mean group size:     65.2                                            


                              Coef. Std.Err.       z  P>|z|  [0.025  0.975]
Intercept                     0.010    0.003   2.977  0.003   0.004   0.017
experience_level[T.Novel]    -0.009    0.003  -2.666  0.008  -0.015  -0.002
experience_level[T.Novel +]   0.001    0.003   0.345  0.730  -0.005   0.008
mouse_id Var                  0.000    0.001                               



Vip Inhibitory





c:\Users\marinag\Anaconda3\envs\visual_behavior_sdk_new\lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
c:\Users\marinag\Anaconda3\envs\visual_behavior_sdk_new\lib\site-packages\statsmodels\base\model.py:604: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\marinag\Anaconda3\envs\visual_behavior_sdk_new\lib\site-packages\statsmodels\regression\mixed_linear_model.py:2200: ConvergenceWarning: Retrying MixedLM optimization with lbfgs
  warnings.warn(
c:\Users\marinag\Anaconda3\envs\visual_behavior_sdk_new\lib\site-packages\statsmodels\base\model.py:604: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\marinag\Anaconda3\envs\visual_behav

Models fitted using Maximum Likelihood (ML) estimation.

--- LRT: Random Intercept (B) vs. Fixed Effects (A) ---
LRT Statistic ($\chi^2$)    7.782387e+01
Degrees of Freedom (df)     1.000000e+00
P-value                     1.126508e-18

--- LRT: Random Intercept + Slope (C) vs. Random Intercept (B) ---
LRT Statistic ($\chi^2$)   -62.20369
Degrees of Freedom (df)      2.00000
P-value                      1.00000

 using model B


                   0        1                    2                      3
0             Model:  MixedLM  Dependent Variable:  mean_response_changes
1  No. Observations:     2277              Method:                   REML
2        No. Groups:       17               Scale:                 0.0010
3   Min. group size:       13      Log-Likelihood:              4558.0959
4   Max. group size:      369           Converged:                    Yes
5   Mean group size:    133.9                                            


                              Coef. Std.Err.   

c:\Users\marinag\Anaconda3\envs\visual_behavior_sdk_new\lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


In [49]:
metric = 'mean_response_changes'
tmp = metrics_table[['cell_specimen_id',  'mouse_id', 'experience_level', 'cell_type', 'project_code']+[metric]]

for cell_type in cell_types: 
    for project_code in metrics_table.project_code.unique():
        print('\n\n\n', cell_type, project_code, '\n\n\n')

        data = tmp[(tmp.cell_type==cell_type) & (tmp.project_code==project_code)]
        stats_dict = run_mixed_linear_model_stats_for_metric(data, metric)




 Excitatory VisualBehavior 





c:\Users\marinag\Anaconda3\envs\visual_behavior_sdk_new\lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
c:\Users\marinag\Anaconda3\envs\visual_behavior_sdk_new\lib\site-packages\statsmodels\base\model.py:604: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\marinag\Anaconda3\envs\visual_behavior_sdk_new\lib\site-packages\statsmodels\regression\mixed_linear_model.py:2200: ConvergenceWarning: Retrying MixedLM optimization with lbfgs
  warnings.warn(
c:\Users\marinag\Anaconda3\envs\visual_behavior_sdk_new\lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
c:\Users\marinag\Anaconda3\envs\visual_behavior_sdk_new\lib

Models fitted using Maximum Likelihood (ML) estimation.

--- LRT: Random Intercept (B) vs. Fixed Effects (A) ---
LRT Statistic ($\chi^2$)     6.099650e+02
Degrees of Freedom (df)      1.000000e+00
P-value                     1.138574e-134

--- LRT: Random Intercept + Slope (C) vs. Random Intercept (B) ---
LRT Statistic ($\chi^2$)   -2.477544
Degrees of Freedom (df)     2.000000
P-value                     1.000000

 using model B


c:\Users\marinag\Anaconda3\envs\visual_behavior_sdk_new\lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)




                   0        1                    2                      3
0             Model:  MixedLM  Dependent Variable:  mean_response_changes
1  No. Observations:     6094              Method:                   REML
2        No. Groups:       13               Scale:                 0.0000
3   Min. group size:      175      Log-Likelihood:             27356.9529
4   Max. group size:      788           Converged:                    Yes
5   Mean group size:    468.8                                            


                              Coef. Std.Err.       z  P>|z|  [0.025 0.975]
Intercept                     0.001    0.000   4.993  0.000   0.001  0.002
experience_level[T.Novel]     0.000    0.000   3.707  0.000   0.000  0.000
experience_level[T.Novel +]  -0.000    0.000  -0.271  0.787  -0.000  0.000
mouse_id Var                  0.000    0.000                              



 Excitatory VisualBehaviorTask1B 





...

c:\Users\marinag\Anaconda3\envs\visual_behavior_sdk_new\lib\site-packages\statsmodels\base\model.py:604: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\marinag\Anaconda3\envs\visual_behavior_sdk_new\lib\site-packages\statsmodels\regression\mixed_linear_model.py:2200: ConvergenceWarning: Retrying MixedLM optimization with lbfgs
  warnings.warn(
c:\Users\marinag\Anaconda3\envs\visual_behavior_sdk_new\lib\site-packages\statsmodels\base\model.py:604: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\marinag\Anaconda3\envs\visual_behavior_sdk_new\lib\site-packages\statsmodels\regression\mixed_linear_model.py:2200: ConvergenceWarning: Retrying MixedLM optimization with cg
  warnings.warn(
c:\Users\marinag\Anaconda3\envs\visual_behavior_sdk_new\lib\site-packages\statsmodels

Models fitted using Maximum Likelihood (ML) estimation.

--- LRT: Random Intercept (B) vs. Fixed Effects (A) ---
LRT Statistic ($\chi^2$)   -6.815753e-08
Degrees of Freedom (df)     1.000000e+00
P-value                     1.000000e+00

--- LRT: Random Intercept + Slope (C) vs. Random Intercept (B) ---
LRT Statistic ($\chi^2$)   -7.961244
Degrees of Freedom (df)     2.000000
P-value                     1.000000

 using model A


           Intercept  experience_level[T.Novel]  experience_level[T.Novel +]
Coef.       0.001102                   0.001663                    -0.000116
Std. Err.   0.000305                   0.000421                     0.000435
t           3.608260                   3.948091                    -0.265962
P>|t|       0.000379                   0.000105                     0.790509
Lower CI    0.000500                   0.000833                    -0.000972
Upper CI    0.001703                   0.002493                     0.000741





 Vip Inhibitory VisualB

c:\Users\marinag\Anaconda3\envs\visual_behavior_sdk_new\lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
c:\Users\marinag\Anaconda3\envs\visual_behavior_sdk_new\lib\site-packages\statsmodels\base\model.py:604: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\marinag\Anaconda3\envs\visual_behavior_sdk_new\lib\site-packages\statsmodels\regression\mixed_linear_model.py:2200: ConvergenceWarning: Retrying MixedLM optimization with lbfgs
  warnings.warn(
c:\Users\marinag\Anaconda3\envs\visual_behavior_sdk_new\lib\site-packages\statsmodels\base\model.py:604: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\marinag\Anaconda3\envs\visual_behav

Models fitted using Maximum Likelihood (ML) estimation.

--- LRT: Random Intercept (B) vs. Fixed Effects (A) ---
LRT Statistic ($\chi^2$)    16.878178
Degrees of Freedom (df)      1.000000
P-value                      0.000040

--- LRT: Random Intercept + Slope (C) vs. Random Intercept (B) ---
LRT Statistic ($\chi^2$)   -10.085954
Degrees of Freedom (df)      2.000000
P-value                      1.000000

 using model B


                   0        1                    2                      3
0             Model:  MixedLM  Dependent Variable:  mean_response_changes
1  No. Observations:     1578              Method:                   REML
2        No. Groups:        6               Scale:                 0.0015
3   Min. group size:      142      Log-Likelihood:              2880.4581
4   Max. group size:      369           Converged:                    Yes
5   Mean group size:    263.0                                            


                              Coef. Std.Err.       z 

c:\Users\marinag\Anaconda3\envs\visual_behavior_sdk_new\lib\site-packages\statsmodels\base\model.py:604: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\marinag\Anaconda3\envs\visual_behavior_sdk_new\lib\site-packages\statsmodels\regression\mixed_linear_model.py:2206: ConvergenceWarning: MixedLM optimization failed, trying a different optimizer may help.
  warnings.warn(msg, ConvergenceWarning)
c:\Users\marinag\Anaconda3\envs\visual_behavior_sdk_new\lib\site-packages\statsmodels\regression\mixed_linear_model.py:2218: ConvergenceWarning: Gradient optimization failed, |grad| = 8.469191
  warnings.warn(msg, ConvergenceWarning)
c:\Users\marinag\Anaconda3\envs\visual_behavior_sdk_new\lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
c:\Users\marina